In [1]:
import sys
import pathlib
import pandas as pd

PROJECT = pathlib.Path.cwd().parent          
ROOT = PROJECT / "data" / "raw"
SRC = PROJECT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

files = [p for p in ROOT.rglob("*") if p.is_file()]
print(len(files), "files")

344 files


In [15]:
summary = pd.DataFrame({
    "folder": [p.parent.name for p in files],
    "type": ["".join(p.suffixes) for p in files],
})
print(summary.value_counts())

folder       type     
collections  .csv.gz      108
ahmedabad    .csv.gz       36
bengaluru    .csv.gz       36
hyderabad    .csv.gz       36
pune         .csv.gz       36
chennai      .csv.gz       33
jaipur       .csv.gz       33
payments     .csv.gz       28
invoices     .csv.gz        7
reference    .csv           3
chennai      .jsonl.gz      3
jaipur       .jsonl.gz      3
patients     .csv.gz        1
bookings                    1
plan         .csv           1
campaigns    .csv.gz        1
             .csv           1
results                     1
nova                        1
legacy                      1
Name: count, dtype: int64


### Undersatnding the data in one file of Legacy folder 

In [16]:
from kalpa import loaders
legacy_dir = ROOT / "bookings" / "legacy"

In [17]:
one_legacy_file = sorted(legacy_dir.rglob("*.csv.gz"))[0]

In [18]:
df = loaders.read_legacy_file(one_legacy_file)
print(one_legacy_file.relative_to(ROOT))

bookings/legacy/ahmedabad/bookings_2025-01.csv.gz


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12813 entries, 0 to 12812
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   booking_id    12813 non-null  string        
 1   created_at    12813 non-null  datetime64[us]
 2   slot_at       12813 non-null  datetime64[us]
 3   city          12813 non-null  category      
 4   clinic_id     12813 non-null  category      
 5   channel       12813 non-null  category      
 6   source        12813 non-null  category      
 7   patient_id    12813 non-null  string        
 8   patient_name  12813 non-null  string        
 9   test_code     12796 non-null  category      
 10  test_price    12813 non-null  Int32         
 11  home_fee      12813 non-null  Int32         
 12  source_file   12813 non-null  str           
 13  system        12813 non-null  str           
dtypes: Int32(2), category(5), datetime64[us](2), str(2), string(3)
memory usage: 889.6 KB


In [20]:
df.head()

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,bookings_2025-01.csv.gz,legacy
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,bookings_2025-01.csv.gz,legacy
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,bookings_2025-01.csv.gz,legacy
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,bookings_2025-01.csv.gz,legacy
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,bookings_2025-01.csv.gz,legacy


##### Check if all the legacy files are same or not 

In [21]:
import gzip

legacy_paths = sorted((ROOT / "bookings" / "legacy").rglob("*.csv.gz"))

headers = set()

for p in legacy_paths:
    with gzip.open(p, "rt", encoding="latin-1") as f:
        first_line = f.readline().strip()
    headers.add(first_line)

print(len(legacy_paths), "files")
print(len(headers), "different header(s)")

for h in headers:
    print(h)

102 files
1 different header(s)
booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee


In [22]:
legacy_files = sorted((ROOT / "bookings" / "legacy").rglob("*.csv.gz"))
print(pd.Series([p.parent.name for p in legacy_files]).value_counts())

print()
all_city = [p for p in ROOT.rglob("*.csv.gz") if p.parent.name in ("ahmedabad","bengaluru","hyderabad","pune","chennai","jaipur")]
print(pd.Series([str(p.parent.parent.relative_to(ROOT)) for p in all_city]).value_counts())

ahmedabad    18
bengaluru    18
hyderabad    18
pune         18
chennai      15
jaipur       15
Name: count, dtype: int64

reference/results    108
bookings/legacy      102
Name: count, dtype: int64


In [23]:
rows = []
for p in legacy_files:
    d = loaders.read_legacy_file(p)
    rows.append({
        "file": str(p.relative_to(ROOT)),
        "rows": len(d),
        "repeated_headers": d.attrs["repeated_header_lines"],
        "bad_dates": int(d.created_at.isna().sum() + d.slot_at.isna().sum()),
        "cities": ",".join(sorted(map(str, d.city.dropna().unique()))),
    })

per_file = pd.DataFrame(rows)

print("files with repeated headers:", (per_file.repeated_headers > 0).sum())
print("files with bad dates:", (per_file.bad_dates > 0).sum())
print(per_file.cities.value_counts())

files with repeated headers: 21
files with bad dates: 0
cities
Ahmedabad    18
Hyderabad    18
Pune         18
Chennai      15
Jaipur       15
Bangalore    12
Bengaluru     6
Name: count, dtype: int64


## Merging all the legacy data 

In [24]:
legacy_paths = sorted((ROOT / "bookings" / "legacy").rglob("*.csv.gz"))

legacy = loaders.read_many(legacy_paths, "legacy")
legacy_repeated_headers = legacy.attrs["repeated_header_lines"]  

print(f"{len(legacy_paths)} files -> {len(legacy):,} rows, {legacy.shape[1]} columns")
print("repeated header lines dropped:", legacy_repeated_headers)

# checks that the merge worked
print("rows add up:", len(legacy) == per_file.rows.sum())
print("distinct source files:", legacy.source_file.nunique())


102 files -> 1,252,765 rows, 14 columns
repeated header lines dropped: 21
rows add up: True
distinct source files: 18


#### Created a column called Pairs containing month and city and source_file

In [25]:
legacy["month"] = legacy["source_file"].str.extract(r"(\d{4}-\d{2})")[0]
legacy["pairs"] = legacy["month"] + "_" + legacy["city"].astype(str)

print(legacy["month"].isna().sum(), "rows where the month was not found")   
print(legacy["pairs"].nunique(), "distinct pairs") 

0 rows where the month was not found
102 distinct pairs


In [26]:
legacy.head()

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,month,pairs
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad


In [27]:
legacy.info()

<class 'pandas.DataFrame'>
RangeIndex: 1252765 entries, 0 to 1252764
Data columns (total 16 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   booking_id    1252765 non-null  string        
 1   created_at    1252765 non-null  datetime64[us]
 2   slot_at       1252765 non-null  datetime64[us]
 3   city          1252765 non-null  str           
 4   clinic_id     1252765 non-null  str           
 5   channel       1252765 non-null  category      
 6   source        1252765 non-null  category      
 7   patient_id    1252765 non-null  string        
 8   patient_name  1252765 non-null  string        
 9   test_code     1251475 non-null  category      
 10  test_price    1252765 non-null  Int32         
 11  home_fee      1252765 non-null  Int32         
 12  source_file   1252765 non-null  str           
 13  system        1252765 non-null  str           
 14  month         1252765 non-null  str           
 15  pairs    

## Cleaning the Legacy data 

In [28]:
print(legacy.shape)

(1252765, 16)


In [29]:
print(legacy.isna().sum())

booking_id         0
created_at         0
slot_at            0
city               0
clinic_id          0
channel            0
source             0
patient_id         0
patient_name       0
test_code       1290
test_price         0
home_fee           0
source_file        0
system             0
month              0
pairs              0
dtype: int64


#### Checking the missing points in Test_code

In [30]:
missing = legacy[legacy.test_code.isna()]

print(missing.city.value_counts())
print(missing.month.value_counts().sort_index())

missing.head(10)

city
Hyderabad    255
Ahmedabad    250
Pune         202
Chennai      192
Bangalore    170
Bengaluru    118
Jaipur       103
Name: count, dtype: int64
month
2025-01    84
2025-02    76
2025-03    56
2025-04    82
2025-05    66
2025-06    58
2025-07    73
2025-08    64
2025-09    71
2025-10    77
2025-11    67
2025-12    73
2026-01    82
2026-02    86
2026-03    96
2026-04    68
2026-05    52
2026-06    59
Name: count, dtype: int64


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,month,pairs
1023,BK-AHM-202501-000462,2025-01-13 00:26:00,2025-01-13 08:25:00,Ahmedabad,AHM-01,clinic,walkin,PAHM0021638,Manish Sharma,NaN,550,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
1256,BK-AHM-202501-000572,2025-01-14 06:37:00,2025-01-18 13:08:00,Ahmedabad,AHM-03,clinic,phone,PAHM0001702,Renée Pillai,NaN,1100,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
2242,BK-AHM-202501-001008,2025-01-21 04:20:00,2025-01-26 11:50:00,Ahmedabad,AHM-05,home,app,PAHM0028768,Amit Fernandes,NaN,2999,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
4963,BK-AHM-202501-002213,2025-01-07 10:26:00,2025-01-09 17:32:00,Ahmedabad,AHM-03,clinic,app,PAHM0020634,Priya Desai,NaN,120,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
5116,BK-AHM-202501-002281,2025-01-01 07:22:00,2025-01-03 07:54:00,Ahmedabad,AHM-03,home,app,PAHM0007861,Farhan Rao,NaN,800,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
5499,BK-AHM-202501-002449,2025-01-15 07:31:00,2025-01-16 13:25:00,Ahmedabad,AHM-05,home,app,PAHM0021061,Rohan Joshi,NaN,550,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
5560,BK-AHM-202501-002476,2025-01-10 05:56:00,2025-01-10 13:40:00,Ahmedabad,AHM-01,clinic,phone,PAHM0023151,Vihaan Iyer,NaN,150,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
5960,BK-AHM-202501-002667,2025-01-12 09:43:00,2025-01-15 16:54:00,Ahmedabad,AHM-03,clinic,app,PAHM0014931,Priya Desai,NaN,1100,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
6562,BK-AHM-202501-002924,2025-01-13 03:06:00,2025-01-17 08:03:00,Ahmedabad,AHM-01,clinic,phone,PAHM0021010,Priya Fernandes,NaN,550,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
7211,BK-AHM-202501-003206,2025-01-16 01:01:00,2025-01-16 09:32:00,Ahmedabad,AHM-02,home,app,PAHM0030293,Meera Kulkarni,NaN,2999,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad


#### Trying to fill the test code with patterns 

In [31]:
known = legacy[legacy.test_code.notna()]
price_to_codes = known.groupby("test_price", observed=True).test_code.nunique()

missing = legacy[legacy.test_code.isna()]
m = missing.test_price.map(price_to_codes)

print("missing rows whose price maps to exactly 1 code:", (m == 1).sum())
print("maps to several codes:", (m > 1).sum())
print("price never seen with a code:", m.isna().sum())

missing rows whose price maps to exactly 1 code: 822
maps to several codes: 468
price never seen with a code: 0


### Droping  Test Codes

In [ ]:
# price -> number of distinct codes, and the code itself (from rows that have one)
grp = known.groupby("test_price", observed=True).test_code
codes_per_price = grp.nunique()
single_code = grp.first()

is_missing = legacy.test_code.isna()
n_codes = legacy.test_price.map(codes_per_price)

fill_mask = is_missing & (n_codes == 1)
drop_mask = is_missing & (n_codes > 1)


print(fill_mask.sum(), drop_mask.sum())   

# fill the 822
legacy.loc[fill_mask, "test_code"] = legacy.loc[fill_mask, "test_price"].map(single_code)

# drop the 468
legacy = legacy[~drop_mask].copy()

print("remaining missing test_code:", legacy.test_code.isna().sum())   # expect 0
print("rows now:", len(legacy))

822 468
remaining missing test_code: 0
rows now: 1252297


#### Checking City

In [33]:
legacy['city'].unique()

<StringArray>
['Ahmedabad', 'Bangalore', 'Bengaluru', 'Chennai', 'Hyderabad', 'Jaipur',
 'Pune']
Length: 7, dtype: str

In [34]:
legacy['month'] = legacy['source_file'].str.extract(r"(\d{4}-\d{2})")[0]

blr = legacy[legacy['city'].isin(['Bangalore', 'Bengaluru'])]

print(blr.groupby('city')['month'].agg(['min', 'max']))
print(blr.groupby('city')['clinic_id'].unique())

               min      max
city                       
Bangalore  2025-01  2025-12
Bengaluru  2026-01  2026-06
city
Bangalore    [BLR-05, BLR-02, BLR-03, BLR-01, BLR-04, BLR-06]
Bengaluru    [BLR-04, BLR-03, BLR-06, BLR-02, BLR-05, BLR-01]
Name: clinic_id, dtype: object


In [35]:
# Normalise the city name (Bangalore -> Bengaluru)
legacy["city"] = (
    legacy["city"]
    .astype("string")
    .str.strip()
    .replace({"Bangalore": "Bengaluru"})
)

# pairs was built from the old spelling, so rebuild it
legacy["pairs"] = legacy["month"] + "_" + legacy["city"]

# Verify
print(sorted(legacy["city"].unique()))
print((legacy["city"] == "Bangalore").sum(), "rows still say Bangalore")
print(legacy["pairs"].nunique(), "distinct pairs")

['Ahmedabad', 'Bengaluru', 'Chennai', 'Hyderabad', 'Jaipur', 'Pune']
0 rows still say Bangalore
102 distinct pairs


### Comparing created at & slot at

In [36]:
print(legacy[['created_at', 'slot_at']].dtypes)
print(legacy[['created_at', 'slot_at']].sample(10, random_state=0))
print(legacy[['created_at', 'slot_at']].isna().sum())

created_at    datetime64[us]
slot_at       datetime64[us]
dtype: object
                 created_at             slot_at
570678  2025-02-26 03:50:00 2025-03-02 13:38:00
86423   2025-07-11 10:23:00 2025-07-14 10:38:00
459726  2026-02-18 00:01:00 2026-02-20 07:16:00
1121474 2025-06-15 04:51:00 2025-06-17 09:56:00
971834  2025-01-31 06:57:00 2025-02-02 11:42:00
190279  2026-03-07 04:53:00 2026-03-09 09:26:00
419952  2025-12-05 02:26:00 2025-12-05 09:34:00
1034927 2025-11-23 04:40:00 2025-11-23 13:58:00
210308  2026-05-13 08:24:00 2026-05-18 11:36:00
758692  2025-02-19 01:09:00 2025-02-24 08:50:00
created_at    0
slot_at       0
dtype: int64


In [37]:
legacy['created_at'] = pd.to_datetime(legacy['created_at'], errors='coerce')
legacy['slot_at']    = pd.to_datetime(legacy['slot_at'], errors='coerce')

after = legacy['created_at'] > legacy['slot_at']
print(after.sum(), "of", len(legacy), "rows have created_at after slot_at")

2442 of 1252297 rows have created_at after slot_at


In [38]:
after = legacy['created_at'] > legacy['slot_at']

dropped = legacy[after].copy()
legacy = legacy[~after].reset_index(drop=True)

print(len(dropped), "rows moved to dropped")
print(len(legacy), "rows remaining in legacy")

2442 rows moved to dropped
1249855 rows remaining in legacy


# Nova folder 

### Loading the Nova data 

#### Finding the files

In [39]:
import gzip, json
import pandas as pd

nova_paths = sorted((ROOT / "bookings" / "nova").rglob("*.jsonl.gz"))
nova_paths = sorted((ROOT / "bookings" / "nova").rglob("*.jsonl.gz"))

print(len(nova_paths), "files")

6 files


#### Finding the structure

In [40]:
with gzip.open(nova_paths[0], "rt", encoding="utf-8") as f:
    first = json.loads(next(f))
print(json.dumps(first, indent=2, ensure_ascii=False))

{
  "booking_id": "nova_che_0087556",
  "created_at": "2026-04-21T01:57:00+05:30",
  "slot_at": "2026-04-25T10:16:00+05:30",
  "city": "Chennai",
  "clinic_id": "CHE-01",
  "channel": "CLINIC",
  "source": "WALKIN",
  "patient": {
    "patient_id": "PCHE0044922",
    "is_new": false
  },
  "tests": [
    {
      "code": "T007",
      "price": 550
    }
  ],
  "home_fee": 0,
  "rescheduled_from": null,
  "status_at_export": "booked"
}


#### Info of raw data

In [41]:
frames = []
for p in nova_paths:
    with gzip.open(p, "rt", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    d = pd.json_normalize(records)
    d["source_file"] = p.name
    d["system"] = "nova"
    frames.append(d)

nova_raw = pd.concat(frames, ignore_index=True)
print(nova_raw.shape)
# here it is one row per booking 
nova_raw.info()
print(nova_raw.head(3).T)

(26543, 15)
<class 'pandas.DataFrame'>
RangeIndex: 26543 entries, 0 to 26542
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   booking_id          26543 non-null  str   
 1   created_at          26543 non-null  str   
 2   slot_at             26543 non-null  str   
 3   city                26543 non-null  str   
 4   clinic_id           26543 non-null  str   
 5   channel             26543 non-null  str   
 6   source              26543 non-null  str   
 7   tests               26543 non-null  object
 8   home_fee            26543 non-null  int64 
 9   rescheduled_from    1500 non-null   str   
 10  status_at_export    26543 non-null  str   
 11  patient.patient_id  26543 non-null  str   
 12  patient.is_new      26543 non-null  bool  
 13  source_file         26543 non-null  str   
 14  system              26543 non-null  str   
dtypes: bool(1), int64(1), object(1), str(12)
memory usage: 2.9+ MB
      

### Making it one row per test

In [42]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from kalpa import loaders


nova_paths = sorted((ROOT / "bookings" / "nova").rglob("*.jsonl.gz"))

nova = loaders.read_many(nova_paths, "nova")

print(f"{len(nova_paths)} files -> {len(nova):,} rows, {nova.shape[1]} columns")
print(nova.groupby("source_file").size())

6 files -> 62,418 rows, 16 columns
source_file
bookings_2026-04.jsonl.gz    20704
bookings_2026-05.jsonl.gz    20841
bookings_2026-06.jsonl.gz    20873
dtype: int64


In [43]:
nova

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,rescheduled_from,status_at_export,source_file,system
0,nova_che_0087556,2026-04-21 01:57:00,2026-04-25 10:16:00,Chennai,CHE-01,clinic,walkin,PCHE0044922,<NA>,T007,550,0,NaN,booked,bookings_2026-04.jsonl.gz,nova
1,nova_che_0087557,2026-03-28 03:02:00,2026-04-01 09:17:00,Chennai,CHE-03,clinic,phone,PCHE0047035,<NA>,T007,550,0,NaN,booked,bookings_2026-04.jsonl.gz,nova
2,nova_che_0087557,2026-03-28 03:02:00,2026-04-01 09:17:00,Chennai,CHE-03,clinic,phone,PCHE0047035,<NA>,T011,750,0,NaN,booked,bookings_2026-04.jsonl.gz,nova
3,nova_che_0087557,2026-03-28 03:02:00,2026-04-01 09:17:00,Chennai,CHE-03,clinic,phone,PCHE0047035,<NA>,T004,650,0,NaN,booked,bookings_2026-04.jsonl.gz,nova
4,nova_che_0087558,2026-04-28 04:46:00,2026-04-30 08:19:00,Chennai,CHE-05,clinic,phone,PCHE0051652,<NA>,T004,650,0,NaN,booked,bookings_2026-04.jsonl.gz,nova
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62413,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,<NA>,T007,550,150,nova_jai_0052790,booked,bookings_2026-06.jsonl.gz,nova
62414,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,<NA>,T006,650,150,nova_jai_0052790,booked,bookings_2026-06.jsonl.gz,nova
62415,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,<NA>,T006,650,0,nova_jai_0054062,booked,bookings_2026-06.jsonl.gz,nova
62416,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,<NA>,T002,120,0,nova_jai_0054062,booked,bookings_2026-06.jsonl.gz,nova


#### Checking the reschedule data points 

In [44]:
rescheduled_nova=nova.loc[nova['rescheduled_from'].notna()]

In [45]:
rescheduled_nova


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,rescheduled_from,status_at_export,source_file,system
13053,nova_che_0093125,2026-04-12 21:05:00,2026-04-20 06:32:00,Chennai,CHE-01,clinic,phone,PCHE0048651,<NA>,T010,150,0,nova_che_0090017,booked,bookings_2026-04.jsonl.gz,nova
13054,nova_che_0093126,2026-04-11 04:33:00,2026-04-13 06:14:00,Chennai,CHE-03,clinic,phone,PCHE0053174,<NA>,T009,800,0,nova_che_0092841,booked,bookings_2026-04.jsonl.gz,nova
13055,nova_che_0093126,2026-04-11 04:33:00,2026-04-13 06:14:00,Chennai,CHE-03,clinic,phone,PCHE0053174,<NA>,T003,450,0,nova_che_0092841,booked,bookings_2026-04.jsonl.gz,nova
13056,nova_che_0093127,2026-04-04 07:17:00,2026-04-09 09:58:00,Chennai,CHE-02,home,app,PCHE0045471,<NA>,T013,400,150,nova_che_0088334,booked,bookings_2026-04.jsonl.gz,nova
13057,nova_che_0093127,2026-04-04 07:17:00,2026-04-09 09:58:00,Chennai,CHE-02,home,app,PCHE0045471,<NA>,T007,550,150,nova_che_0088334,booked,bookings_2026-04.jsonl.gz,nova
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62413,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,<NA>,T007,550,150,nova_jai_0052790,booked,bookings_2026-06.jsonl.gz,nova
62414,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,<NA>,T006,650,150,nova_jai_0052790,booked,bookings_2026-06.jsonl.gz,nova
62415,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,<NA>,T006,650,0,nova_jai_0054062,booked,bookings_2026-06.jsonl.gz,nova
62416,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,<NA>,T002,120,0,nova_jai_0054062,booked,bookings_2026-06.jsonl.gz,nova


In [46]:
originals = nova.loc[nova["status_at_export"].eq("rescheduled"), "booking_id"].unique()
replaced_from = nova.loc[nova["rescheduled_from"].notna(), "rescheduled_from"].unique()

print("original bookings being dropped:", len(originals))
print("bookings referenced as rescheduled_from:", len(replaced_from))
print("originals with no replacement found in nova:", len(set(originals) - set(replaced_from)))
print("rescheduled_from pointers with no matching original:", len(set(replaced_from) - set(originals)))

original bookings being dropped: 1499
bookings referenced as rescheduled_from: 1500
originals with no replacement found in nova: 0
rescheduled_from pointers with no matching original: 1


### Dropping the rescheduled once

In [47]:
is_rescheduled = nova["status_at_export"].eq("rescheduled")

# originals being removed — kept here for reference/audit, not merged
nova_dropped_originals = nova.loc[is_rescheduled].copy()

# what goes forward into harmonise() / the merge
nova_current = (
    nova.loc[~is_rescheduled]
    .drop(columns=["rescheduled_from", "status_at_export"])
    .reset_index(drop=True)
)

print("dropped (originals):", len(nova_dropped_originals),
      "| bookings:", nova_dropped_originals.booking_id.nunique())
print("nova_current rows:", len(nova_current))

dropped (originals): 3541 | bookings: 1499
nova_current rows: 58877


In [48]:
legacy_cols, nova_cols = set(legacy.columns), set(nova_current.columns)

print("only in legacy:    ", sorted(legacy_cols - nova_cols))
print("only in nova:      ", sorted(nova_cols - legacy_cols))   # expect []
print("dtypes that differ on shared columns:")
for c in sorted(legacy_cols & nova_cols):
    if str(legacy[c].dtype) != str(nova_current[c].dtype):
        print(f"  {c:12s} legacy={legacy[c].dtype!s:16s} nova={nova_current[c].dtype}")

only in legacy:     ['month', 'pairs']
only in nova:       []
dtypes that differ on shared columns:
  booking_id   legacy=string           nova=str
  channel      legacy=category         nova=str
  city         legacy=string           nova=str
  home_fee     legacy=Int32            nova=int64
  patient_id   legacy=string           nova=str
  patient_name legacy=string           nova=object
  source       legacy=category         nova=str
  test_code    legacy=category         nova=str
  test_price   legacy=Int32            nova=int64


## Merging

In [49]:
FINAL_COLS = [
    "booking_id", "created_at", "slot_at", "city", "clinic_id", "channel",
    "source", "patient_id", "patient_name", "test_code", "test_price",
    "home_fee", "source_file", "system",
]

def harmonise(df):
    df = df.copy()
    for c in FINAL_COLS:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[FINAL_COLS].copy()

    for c in ["created_at", "slot_at"]:
        df[c] = df[c].astype("datetime64[us]")
    for c in ["booking_id", "patient_id", "patient_name"]:
        df[c] = df[c].astype("string")
    for c in ["city", "clinic_id", "channel", "source", "test_code"]:
        df[c] = df[c].astype("string").str.strip()

    df["city"]    = df["city"].replace({"Bangalore": "Bengaluru"})
    df["channel"] = df["channel"].str.lower()
    df["source"]  = df["source"].str.lower()

    for c in ["test_price", "home_fee"]:
        df[c] = df[c].astype("Int32")

    df["month"] = df["source_file"].str.extract(r"(\d{4}-\d{2})")[0]
    df["pairs"] = df["month"] + "_" + df["city"]
    return df

legacy_h = harmonise(legacy)
nova_h   = harmonise(nova_current)

overlap = sorted(set(legacy_h.pairs) & set(nova_h.pairs))
print("overlapping city-month pairs:", overlap if overlap else "none")

bookings = pd.concat([legacy_h, nova_h], ignore_index=True)

for c in ["city", "clinic_id", "channel", "source", "system", "test_code"]:
    bookings[c] = bookings[c].astype("category")

print(f"rows: {len(bookings):,} -> expect", len(legacy_h) + len(nova_h))
print("cities:", sorted(bookings.city.cat.categories))
print("date range:", bookings.created_at.min(), "->", bookings.created_at.max())
print("\nnulls:")
print(bookings.isna().sum()[lambda s: s > 0])

overlapping city-month pairs: none
rows: 1,308,732 -> expect 1308732
cities: ['Ahmedabad', 'Bengaluru', 'Chennai', 'Hyderabad', 'Jaipur', 'Pune']
date range: 2024-12-26 21:08:00 -> 2026-06-30 17:51:00

nulls:
patient_name    58877
dtype: int64


In [50]:
bookings

,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system,month,pairs
0,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T009,800,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
1,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
2,BK-AHM-202501-000001,2025-01-08 03:16:00,2025-01-09 08:26:00,Ahmedabad,AHM-03,clinic,walkin,PAHM0028525,Saanvi Bose,T007,550,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
3,BK-AHM-202501-000002,2025-01-03 05:18:00,2025-01-06 13:22:00,Ahmedabad,AHM-04,clinic,app,PAHM0031384,Nisha Iyer,T004,650,0,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
4,BK-AHM-202501-000003,2025-01-27 10:46:00,2025-01-28 13:37:00,Ahmedabad,AHM-01,home,app,PAHM0027001,Sneha Menon,T003,450,150,bookings_2025-01.csv.gz,legacy,2025-01,2025-01_Ahmedabad
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308727,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,<NA>,T007,550,150,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur
1308728,nova_jai_0055752,2026-06-07 02:29:00,2026-06-12 11:18:00,Jaipur,JAI-02,home,phone,PJAI0017296,<NA>,T006,650,150,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur
1308729,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,<NA>,T006,650,0,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur
1308730,nova_jai_0055753,2026-06-05 07:51:00,2026-06-06 10:25:00,Jaipur,JAI-03,clinic,phone,PJAI0027566,<NA>,T002,120,0,bookings_2026-06.jsonl.gz,nova,2026-06,2026-06_Jaipur


In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12813 entries, 0 to 12812
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   booking_id    12813 non-null  string        
 1   created_at    12813 non-null  datetime64[us]
 2   slot_at       12813 non-null  datetime64[us]
 3   city          12813 non-null  category      
 4   clinic_id     12813 non-null  category      
 5   channel       12813 non-null  category      
 6   source        12813 non-null  category      
 7   patient_id    12813 non-null  string        
 8   patient_name  12813 non-null  string        
 9   test_code     12796 non-null  category      
 10  test_price    12813 non-null  Int32         
 11  home_fee      12813 non-null  Int32         
 12  source_file   12813 non-null  str           
 13  system        12813 non-null  str           
dtypes: Int32(2), category(5), datetime64[us](2), str(2), string(3)
memory usage: 889.6 KB


## Exporting the data to csv

In [ ]:


OUT = ROOT.parent / "processed"
OUT.mkdir(exist_ok=True)

bookings.to_csv(OUT / "bookings_merged.csv", index=False)

# verify the file
import os
size_mb = os.path.getsize(OUT / "bookings_merged.csv") / 1e6
print(f"saved: {OUT / 'bookings_merged.csv'} | {size_mb:.0f} MB | expected rows: {len(bookings):,}")

saved: /Users/vasudevkamathm/Public/IITGN/Data_analysis/project/project/processed/bookings_merged.csv | 234 MB | expected rows: 1,309,199


In [54]:
CLEAN = ROOT.parent /  "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

bookings.to_csv(CLEAN / "bookings_merged.csv.gz", index=False, compression="gzip")

import os
size_mb = os.path.getsize(CLEAN / "bookings_merged.csv.gz") / 1e6
print(f"saved: {CLEAN / 'bookings_merged.csv.gz'} | {size_mb:.1f} MB | rows: {len(bookings):,}")

saved: /Users/vasudevkamathm/Public/IITGN/Data_analysis/project/project/data/clean/bookings_merged.csv.gz | 16.8 MB | rows: 1,308,732
